In [ ]:
import os
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:
input_file = 'journal_15_oai_dc.xlsx'
input_metadata_path = 'metadata/' + input_file
pdfs_dir_name = input_file.replace('_oai_dc.xlsx', '')

In [ ]:
os.mkdir('pdfs/' + pdfs_dir_name)

In [ ]:
def download_pdf(pdf_tuple):
    url = pdf_tuple[0]
    filename = pdf_tuple[1] + '.pdf'
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"
        }
        response = requests.get(url, headers=headers, timeout=60, verify=False)
    except Exception as e:
        print('Download failed.', pdf_tuple)
        raise Exception(pdf_tuple)
    with open(f'pdfs/{pdfs_dir_name}/' + filename, "wb") as f:
        f.write(response.content)
    return (pdf_tuple, 'Success')


In [ ]:
df = pd.read_excel(input_metadata_path)

pdf_urls = []
for _, row in df.iterrows():
    try:
        pdf_url = row['relation'].split(' | ')
        pdf_url = pdf_url[0]
        pdf_url = pdf_url.replace('/view/', '/download/')

        pdf_name = row['oai_id']
        for ch in [":", "/"]:
            pdf_name = pdf_name.replace(ch, "_")

        pdf_urls.append((pdf_url, pdf_name))
    except:
        with open(f'link_errors_{pdfs_dir_name}.txt', 'a', encoding='utf-8') as txt:
            txt.write(row['oai_id'] + '\n')


In [ ]:
success_results = []
errors = []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(download_pdf, pdf_tuple) for pdf_tuple in pdf_urls]

    for future in tqdm(as_completed(futures), total=len(futures)):
        try:
            result = future.result()
            if result:
                success_results.append(result)
        except Exception as e:
            errors.append(str(e))